# 01. Classes

**Tier:** Intermediate. **Why you'd use this:** A point on a map needs an x, a y, and a way to return a moved copy without editing the original. A class bundles that state and behaviour.

**After this lesson you can:** build a small class with `__init__`; keep instance data on `self`; avoid a shared mutable class attribute.
About 25 minutes. **Uses:** [Definitions](../05-functions/01-definitions-lesson.ipynb)


## Quick review

1. Why is a mutable default list shared across calls?
2. What does `is` compare?

<details><summary>Answers</summary>

Defaults are evaluated once at `def` time. Object identity.
</details>


## The idea

A class is a factory for objects. `__init__` runs after a new instance is created and stores values on `self`. Methods are functions on the class; lookup binds the instance as the first argument, which is why `self` is written explicitly. Class attributes live on the class and are shared. A mutable class attribute is one shared list for every instance.

<details><summary>Explain more</summary>

`Point(1, 2)` builds an instance, then calls `__init__`. `point.moved(1, 0)` looks up `moved` on the class and calls it with `point` as `self`.
</details>


## See it work

Store a point and return a moved copy.


In [1]:
class Point:
    def __init__(self, x, y):
        # 1. Instance attributes live on self.
        self.x = x
        self.y = y

    def moved(self, dx, dy):
        return Point(self.x + dx, self.y + dy)


origin = Point(0, 0)
moved = origin.moved(2, 3)
print(moved.x, moved.y)
print(origin.x, origin.y)

2 3
0 0


## Check yourself

Answer these before you run the cell.

1. After `Bag.tags.append("a")` on two instances, what do both see?
2. Is `Point.moved` a plain function before it is looked up on an instance?


In [2]:
class Bag:
    tags = []

    def __init__(self, name):
        self.name = name


first = Bag("a")
second = Bag("b")
first.tags.append("x")
print(second.tags)


class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def moved(self, dx, dy):
        return Point(self.x + dx, self.y + dy)


print(type(Point.moved).__name__)

['x']
function


<details><summary>Answers</summary>

Both see `['x']`. Yes. Binding to an instance happens on lookup.
</details>


## Why it works this way

An instance has its own namespace. Assignment through `self.name = value` writes into that instance. Reading looks at the instance first, then the class, then bases along the method resolution order. That is attribute lookup. A bare `tags = []` inside the class body creates a class attribute. Every instance that does not shadow `tags` shares that one list, so `append` is visible everywhere. The fix is to create the list in `__init__` on `self`. Deep dive: [Attribute lookup](../../internals/08-attributes/01-lookup-lesson.ipynb).

Methods are ordinary functions stored on the class. The explicit `self` parameter is the instance that lookup will supply. Writing `self` makes the dependency obvious and keeps the same function usable unbound when you need it. Calling `Point.moved(point, 1, 0)` and `point.moved(1, 0)` are the same idea once binding has happened. Before binding, `Point.moved` is still a function, which is evidence that methods are not a separate magic kind of object.

Cost of a tiny class is one type object plus one instance dict. The design reason for classes here is bundling state with the operations that use it, not ceremony. Prefer returning a new `Point` from `moved` when the caller might still need the old coordinates. Mutating in place is fine when the object is clearly owned by one place.

Evidence: two bags share `tags` by identity, and `type(Point.moved).__name__` is `function` while a lookup on an instance produces a bound method. That pair of facts explains both the shared-attribute bug and why `self` is written by hand. Class attributes are for shared immutable constants. Instance attributes are for per-object state. Mixing those jobs is how silent cross-talk starts.


In [3]:
class Bag:
    tags = []

    def __init__(self, name):
        self.name = name


first = Bag("a")
second = Bag("b")
first.tags.append("x")
print(first.tags)
print(second.tags)
print(first.tags is second.tags)


class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def moved(self, dx, dy):
        return Point(self.x + dx, self.y + dy)


print(type(Point.moved).__name__)
print(type(Point(0, 0).moved).__name__)

['x']
['x']
True
function
method


## When to use it

Use a class when several operations share the same state. Keep per-instance data on `self`. Put constants on the class if they are immutable. Don't use a mutable class attribute as a default bucket for every instance. Create the list in `__init__` instead. If two instances surprise you by sharing edits, print `is` on the attribute before you blame the caller.


## Common mistake

Putting a list on the class and appending through an instance. Every instance shares that list.


In [4]:
class Bag:
    tags = []

    def __init__(self, name):
        self.name = name


first = Bag("a")
second = Bag("b")
first.tags.append("x")
print(second.tags)

['x']


Create the list on the instance in `__init__` so each bag has its own.


In [5]:
class Bag:
    def __init__(self, name):
        self.name = name
        self.tags = []


first = Bag("a")
second = Bag("b")
first.tags.append("x")
print(first.tags)
print(second.tags)

['x']
[]


## Try it

Change `moved` so it updates `self` in place and returns `self`. When is that API better, and when does returning a new `Point` protect the caller?


## Recap

- Instance attributes live on `self`. Class attributes are shared.
- Methods are functions; lookup binds the instance as `self`.
- Mutable class attributes are a shared-state bug.

**Next:** [practice](01-classes-practice.ipynb)
